In [1]:
import os
import json
# from data_load import PHOLPhysicsDataset
from torch.utils.data import Dataset, DataLoader

In [4]:
!pip install eva-decord


ERROR: Could not find a version that satisfies the requirement eva-decord (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for eva-decord


In [2]:
import os
import json
import re
from typing import List, Dict
import torch
from torch.utils.data import Dataset
from decord import VideoReader, cpu
import matplotlib.colors as mcolors

class PHOLPhysicsDataset(Dataset):
    def __init__(
        self,
        root: str,
        video_transform=None,
        mask_transform=None,
        include_qa: bool = False,
    ):
        super().__init__()
        self.root = root
        all_dirs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
        valid = []
        for d in all_dirs:
            sd = os.path.join(root, d)
            if os.path.isfile(os.path.join(sd, "obj.json")) and \
               os.path.isfile(os.path.join(sd, "simulation_objects.mp4")):
                valid.append(d)

        numeric = sorted([d for d in valid if d.isdigit()], key=lambda x: int(x))
        non_numeric = sorted([d for d in valid if not d.isdigit()])
        self.scenes = numeric + non_numeric

        self.video_transform = video_transform or (lambda x: x)
        self.mask_transform = mask_transform or (lambda x: x)
        self.include_qa = include_qa
        self._collision_re = re.compile(r"collision", re.IGNORECASE)
        self._motion_re = re.compile(r"sliding|rolling|stationary|accelerating|decelerating", re.IGNORECASE)

    def __len__(self):
        return len(self.scenes)

    def _load_json(self, path: str) -> Dict:
        with open(path, "r") as f:
            return json.load(f)

    def _get_physical_props(self, objects: List[Dict]) -> Dict:
        props = {}
        for obj in objects:
            fr = obj.get("friction", "").split()
            shape = obj.get("geom_type", "unknown")
            rgba_str = obj.get("visual", {}).get("rgba", "")
            color = [float(x) for x in rgba_str.split()] if rgba_str else []
            color_name = self.rgba_to_name(color)
            mass = obj.get("mass", None)
            elasticity = obj.get("elasticity", 0.0)
            velocity = obj.get("velocity", [0, 0, 0])
            try:
                mass = float(mass) if mass is not None else 1.0
            except (ValueError, TypeError):
                mass = 1.0
                
            props[obj["id"]] = {
                "mass": mass, 
                "friction": [float(x) for x in fr] if fr else [0.4],
                "elasticity": elasticity,
                "velocity": velocity,
                "position": [float(obj.get(f"init_possition_{ax}", 0)) for ax in ["x", "y"]] + [0],
                "material": obj.get("material", "unknown"),
                "shape": shape,
                "color": color_name,
            }
        return props

    def _compute_collision(self, frames: List[Dict]) -> bool:
        for fr in frames:
            if fr.get("interactions"):
                return True
            for o in fr.get("objects", {}).values():
                for tax in o.get("taxonomy", []):
                    if any(self._collision_re.search(lbl) for lbl in tax.get("labels", [])):
                        return True
        return False

    def _get_taxonomy(self, frames: List[Dict]) -> Dict[str, List[List[str]]]:
        taxonomy = {}
        for fr in frames:
            for obj_id, obj_state in fr.get("objects", {}).items():
                labels = []
                bbox = obj_state.get("bbox", [[0,0],[0,0]])
                if bbox and bbox != [[0, 0], [0, 0]]:
                    for tax_entry in obj_state.get("taxonomy", []):
                        labels.extend(tax_entry.get("labels", []))
                    taxonomy.setdefault(obj_id, []).append(labels)
        return taxonomy

    def _describe_obj(self, p):
        color_name = p.get("color", "unknown color")
        shape = p.get("shape", "object")
        material = p.get("material", "unknown material")
        return f"{color_name} {shape} made of {material}"

    def _identify_collision_pairs(self, annotations, valid_ids):
        collision_pairs = set()
        for frame in annotations.get("frames", []):
            for obj_id, obj_state in frame.get("objects", {}).items():
                if obj_id in valid_ids:
                    if any("collision" in lbl.lower() 
                          for tax in obj_state.get("taxonomy", []) 
                          for lbl in tax.get("labels", [])):
                        other_ids = [oid for oid in frame.get("objects", {}) 
                               if oid != obj_id and oid in valid_ids]
                        for other_id in other_ids:
                            collision_pairs.add(frozenset({obj_id, other_id}))
        return [tuple(pair) for pair in collision_pairs]

    def _make_qa_for_scene(self, scene: Dict) -> List[Dict[str, str]]:
        qas = []
        props = scene["physical_props"]
        taxonomy = scene["taxonomy"]
        collision = scene["has_collisions"]
        annotations = scene.get("annotations", {})

        valid_ids = set(props.keys())

        #  property questions
        # for obj_id, p in props.items():
        #     desc = self._describe_obj(p)
        #     qas.extend([
        #         {
        #             "question": f"What is the mass of the {desc}?",
        #             "answer": f"{p['mass']:.2f} units" if isinstance(p['mass'], float) else str(p['mass'])
        #         },
        #         {
        #             "question": f"What is the friction coefficient of the {desc}?",
        #             "answer": f"{p['friction'][0]:.2f}" if p['friction'] else "unknown"
        #         },
        #         {
        #             "question": f"What material is the {desc} made of?",
        #             "answer": p['material']
        #         }, 
        #         {
        #             "question": f"How does the {p['shape']} shape of {desc} affect its motion?",
        #             "answer": f"The {p['shape']} shape affects motion by {'allowing rolling with less friction' if p['shape'] in ['sphere', 'cylinder'] else 'creating more surface contact and friction'}."
        #         }
        #     ])

        # scene questions
        qas.extend([
            {
                "question": "Do any objects collide in this scene?",
                "answer": "Yes" if collision else "No"
            },
            {
                "question": "How many objects are in this scene?",
                "answer": str(len(props))
            }
        ])

        # # Physics explanation questions
        # taxonomy_explanations = {
        #     "collision": "due to intersecting trajectories while in motion",
        #     "stationary": "because the net force acting on it was zero",
        #     "sliding": "due to insufficient friction to stop its motion",
        #     "rolling": "because it had both translational and rotational motion",
        #     "accelerating": "when an unbalanced force was applied to it",
        #     "decelerating": "due to opposing forces like friction",
        #     "elastic collision": "where both momentum and kinetic energy were conserved",
        #     "inelastic collision": "where momentum was conserved but some kinetic energy was lost"
        # }

        # for obj_id, labels_seq in taxonomy.items():
        #     p = props[obj_id]
        #     desc = self._describe_obj(p)
        #     all_labels = {lbl.lower() for frame_lbls in labels_seq for lbl in frame_lbls}
            
        #     for label in all_labels:
        #         if label in taxonomy_explanations:
        #             qas.append({
        #                 "question": f"Why did the {desc} exhibit {label} behavior?",
        #                 "answer": f"The {desc} exhibited {label} behavior {taxonomy_explanations[label]}."
        #             })

        print("props", props)
        if collision:
            collision_pairs = self._identify_collision_pairs(annotations, valid_ids)
            print("collision_pairs ", collision_pairs)
            for obj1_id, obj2_id in collision_pairs:
                p1, p2 = props.get(obj1_id), props.get(obj2_id)
                desc1 = self._describe_obj(p1)
                desc2 = self._describe_obj(p2)
                # 
                e1 = float(p1.get('elasticity', 0.7))
                e2 = float(p2.get('elasticity', 0.7))
                # coefficient of restitution (COR, or "elasticity")
                cor = (e1 + e2) / 2
                # Kinetic Energy Loss
                percent_ke_lost = (1 - cor ** 2) * 100

                m1, m2 = p1['mass'], p2['mass']
                if m1 > m2:
                    more_mass = desc1
                    less_mass = desc2
                else:
                    more_mass = desc2
                    less_mass = desc1

                qas.extend([
                    {
                        "question": f"During collision between {desc1} and {desc2}, how is momentum distributed?",
                        "answer": (
                            "Momentum is conserved. " 
                            f"The more massive object ({more_mass}) will experience a smaller change in velocity,"
                            f"while the less massive object ({less_mass}) will experience a larger change in velocity."
                        )
                    },
                    {
                        "question": f"What percentage of kinetic energy is lost when {desc1} collides with {desc2}?",
                        "answer": (
                            f"Approximately {percent_ke_lost:.0f}% of kinetic energy is lost, "
                            f"based on an average coefficient of restitution (elasticity) of {cor:.2f} for this collision."
                        )
                    }
                ])


        counterfactual_qas = self._generate_counterfactuals(props, taxonomy, collision, annotations)
        qas.extend(counterfactual_qas)

        motion_qas = self._generate_motion_questions(taxonomy, props)
        qas.extend(motion_qas)

        # temporal_questions = self._generate_temporal_questions(props, annotations, valid_ids)
        # qas.extend(temporal_questions)
    

        return qas

    def _generate_temporal_questions(self, props, annotations, valid_ids):
        questions = []
        trajectories = self._calculate_trajectories(annotations, valid_ids)
        
        # first/last moving object
        moving_objs = [oid for oid in valid_ids if sum(abs(v) for v in props[oid]['velocity']) > 0.1]
        if moving_objs:
            first_mover = min(moving_objs, key=lambda x: trajectories[x]['first_move_frame'])
            last_mover = max(moving_objs, key=lambda x: trajectories[x]['last_move_frame'])
            
            questions.append({
                "question": "Which object was the first to start moving and why?",
                "answer": (
                    f"The {self._describe_obj(props[first_mover])} moved first because "
                    f"{'it was pushed initially' if random.random() > 0.5 else 'it had less friction'}"
                )
            })
        
        return questions

    def _calculate_trajectories(self, annotations, valid_ids):
        trajectories = {}
        for obj_id in valid_ids:
            trajectories[obj_id] = {
                'positions': [],
                'velocities': [],
                'first_move_frame': float('inf'),
                'last_move_frame': -1,
                'max_distance': 0
            }
        
        for frame_idx, frame in enumerate(annotations.get("frames", [])):
            for obj_id, obj_state in frame.get("objects", {}).items():
                if obj_id in valid_ids:
                    pos = obj_state.get("position", [0,0,0])
                    vel = obj_state.get("velocity", [0,0,0])
                    trajectories[obj_id]['positions'].append(pos)
                    trajectories[obj_id]['velocities'].append(vel)
                    
                    if sum(abs(v) for v in vel) > 0.1:  # If moving
                        trajectories[obj_id]['first_move_frame'] = min(
                            trajectories[obj_id]['first_move_frame'], frame_idx)
                        trajectories[obj_id]['last_move_frame'] = max(
                            trajectories[obj_id]['last_move_frame'], frame_idx)
        
        for obj_id in trajectories:
            if len(trajectories[obj_id]['positions']) > 1:
                start = trajectories[obj_id]['positions'][0]
                end = trajectories[obj_id]['positions'][-1]
                trajectories[obj_id]['max_distance'] = sum(
                    (e-s)**2 for s,e in zip(start, end))**0.5
    
        return trajectories
        
    def _generate_motion_questions(self, taxonomy, props):
        motion_qas = []
        # 'constant velocity', 'rolling motion with slipping', 'sliding with friction', 'inelastic collision', 'moving to stopping'
        motion_explanations = {
            "sliding with friction": "occurs when friction is insufficient to initiate rolling",
            "rolling motion": "happens when rotational motion dominates with minimal sliding friction",
            "stationary": "indicates balanced forces with net force of zero",
            "accelerating": "requires an unbalanced force greater than friction",
            "decelerating": "results from opposing forces like friction or air resistance"
        }
        
        for obj_id, labels_seq in taxonomy.items():
            all_labels = {lbl.lower() for frame_lbls in labels_seq for lbl in frame_lbls}
            p = props[obj_id]
            desc = self._describe_obj(p)
            # print("all_labels", all_labels)
            for motion_type in motion_explanations:
                # print("motion_type", motion_type)
                # print(" if motion_type in all_labels",  motion_type in all_labels)
                if motion_type in all_labels:
                    motion_qas.append({
                        "question": f"What physical properties cause the {desc} to exhibit {motion_type} behavior?",
                        "answer": ( 
                            f"The {desc} shows {motion_type} because {motion_explanations[motion_type]}. "
                            f" Specifically, its mass ({p['mass']:.2f}) and friction ({p['friction'][0]:.2f}) contribute to this motion."
                        )
                    })
                    motion_qas.append({
                        "question": f"If the {desc} continues moving under current conditions, what will be its motion state in 5 seconds?",
                        "answer": self._predict_future_motion(motion_type, p)
                    })
        
        return motion_qas

    def _predict_future_motion(self, current_motion, props):
        if current_motion == "accelerating":
            if props['friction'][0] < 0.2:
                return "Will continue accelerating until reaching terminal velocity or encountering an obstacle"
            else:
                return "Will eventually reach constant velocity when acceleration equals friction"
        elif current_motion == "decelerating":
            return "Will come to a complete stop due to energy dissipation"
        elif current_motion == "rolling":
            return "Will gradually slow down due to rolling friction and possibly transition to sliding"
        else:
            return "Will maintain current state unless acted upon by external forces"

    # def _generate_counterfactuals(self, props, taxonomy, had_collision, annotations=None):
    #     counterfactuals = []
    #     obj_ids = list(props.keys())
        
    #     if len(obj_ids) >= 2:
    #         obj1, obj2 = obj_ids[0], obj_ids[1]
    #         p1, p2 = props[obj1], props[obj2]
    #         desc1 = self._describe_obj(p1)
    #         desc2 = self._describe_obj(p2)
            
    #         # mass variation
    #         counterfactuals.append({
    #             "question": f"How would the outcome change if {desc1} was 10x more massive?",
    #             "answer": f"{desc1.capitalize()} would dominate the interaction, showing minimal velocity change while {desc2} would be dramatically accelerated upon collision."
    #         })
            
    #         # material change
    #         counterfactuals.append({
    #             "question": f"What if {desc2} was made of perfectly elastic material?",
    #             "answer": "The collision would conserve nearly all kinetic energy, resulting in near-perfect rebound with minimal energy loss."
    #         })
            
    #         # environmental change
    #         counterfactuals.append({
    #             "question": "How would the scene differ in a microgravity environment?",
    #             "answer": "Objects would maintain constant velocity until collision, with post-collision motion dominated by rotation rather than sliding due to lack of normal force."
    #         })
        
    #     return counterfactuals

    def _generate_counterfactuals(self, props, taxonomy, had_collision, annotations=None):
        counterfactuals = []
        if not annotations:
            return counterfactuals
    
        # Build a timeline of collision events
        events = []
        for frame_idx, frame in enumerate(annotations.get("frames", [])):
            for obj_id, obj_state in frame.get("objects", {}).items():
                for tax_entry in obj_state.get("taxonomy", []):
                    for label in tax_entry.get("labels", []):
                        if "collision" in label.lower():
                            pos = obj_state.get("position", [None, None, None])
                            events.append({
                                "frame": frame_idx,
                                "obj_id": obj_id,
                                "label": label,
                                "pos": pos,
                                "other_ids": [
                                    oid for oid in frame.get("objects", {}) if oid != obj_id
                                ]
                            })
    
        for event in events:
            obj_id = event["obj_id"]
            desc = self._describe_obj(props[obj_id])
            frame_idx = event["frame"]
            pos = event["pos"]
            for other_id in event["other_ids"]:
                other_desc = self._describe_obj(props[other_id])
                # 1. Object removal at collision
                counterfactuals.append({
                    "type": "counterfactual_removal",
                    "question": (
                        f"If the {desc} had not been present at frame {frame_idx}, "
                        f"would the {other_desc} have collided with any other object at position {pos}?"
                    ),
                    "answer": (
                        f"No, without the {desc}, the {other_desc} would not have experienced a collision at frame {frame_idx} and position {pos}, "
                        "potentially altering its subsequent trajectory and interactions."
                    )
                })
                # 2. Property change at collision
                old_mass = props[obj_id]['mass']
                counterfactuals.append({
                    "type": "counterfactual_mass",
                    "question": (
                        f"If the {desc} had a mass of {old_mass*2:.2f} instead of {old_mass:.2f} at the moment of collision in frame {frame_idx}, "
                        f"how would the motion of the {other_desc} have changed?"
                    ),
                    "answer": (
                        f"With double the mass, the {desc} would have changed velocity less and transferred more momentum to the {other_desc}, "
                        "causing the {other_desc} to move faster after the collision."
                    )
                })
                # 3. Material property change
                old_material = props[obj_id]['material']
                counterfactuals.append({
                    "type": "counterfactual_material",
                    "question": (
                        f"If the {desc} was made of rubber instead of {old_material} during its collision at frame {frame_idx}, "
                        f"what would be the outcome for the {other_desc}?"
                    ),
                    "answer": (
                        f"The collision would be more elastic, so the {desc} would bounce more, and the {other_desc} would receive less kinetic energy loss."
                    )
                })
                # 4. Temporal delay
                counterfactuals.append({
                    "type": "counterfactual_delay",
                    "question": (
                        f"If the {desc} had entered the scene 5 frames later, would the collision with the {other_desc} at frame {frame_idx} still have occurred?"
                    ),
                    "answer": (
                        f"No, a later entry would have missed the intersection with the {other_desc}, so the collision at frame {frame_idx} would not have occurred."
                    )
                })
        return counterfactuals


    def rgba_to_name(self, rgba):
        if not rgba or len(rgba) < 3:
            return "unknown color"
        
        rgb = tuple(rgba[:3])
        min_dist = float('inf')
        best_name = "unknown color"
        
        for name, hex_val in mcolors.CSS4_COLORS.items():
            named_rgb = mcolors.to_rgb(hex_val)
            dist = sum((c1 - c2)**2 for c1, c2 in zip(rgb, named_rgb))
            if dist < min_dist:
                min_dist = dist
                best_name = name
                
        return best_name.replace('grey', 'gray').replace('gray', 'grey')  # Standardize spelling

    def __getitem__(self, idx: int) -> Dict:
        sid = self.scenes[idx]
        scene_dir = os.path.join(self.root, sid)
        d = self._load_json(os.path.join(scene_dir, "obj.json"))
        frames = d.get("frames", [])

        valid_ids = {
            obj_id
            for fr in frames
            for obj_id, obj_state in fr.get("objects", {}).items()
            if obj_state.get("bbox", [[0,0],[0,0]]) != [[0, 0], [0, 0]]
        }

        physical_props = self._get_physical_props(d.get("objects", []))
        physical_props = {oid: physical_props[oid] for oid in valid_ids if oid in physical_props}

        has_collisions = self._compute_collision(frames)
        taxonomy = self._get_taxonomy(frames)

        vid = self.video_transform(
            self._read_video(os.path.join(scene_dir, "simulation_objects.mp4")))
        seg = self.mask_transform(
            self._read_video(os.path.join(scene_dir, "simulation_objects_segmentation.mp4")))

        qa = None
        if self.include_qa:
            scene = {
                "physical_props": physical_props,
                "taxonomy": taxonomy,
                "has_collisions": has_collisions,
                "annotations": d,
            }
            qa = self._make_qa_for_scene(scene)

        return {
            "scene_id": sid,
            "video": vid,
            "segmentation": seg,
            "physical_props": physical_props,
            "has_collisions": has_collisions,
            "taxonomy": taxonomy,
            "annotations": d,
            "qa": qa,
        }

    def _read_video(self, path: str) -> torch.Tensor:
        vr = VideoReader(path, ctx=cpu(0))
        frames = vr.get_batch(range(len(vr))).asnumpy()
        return torch.from_numpy(frames).permute(3, 0, 1, 2).float().div(255.0)

ModuleNotFoundError: No module named 'decord'